# OmniLingual ASR 1B Levantine Custom Run
This notebook mirrors the Levantine Whisper run flow using the official OmniLingual 1B LLM-ASR recipe, while reusing the exact same source train/val/test split manifests.

In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path('/home/MohammadNabulsi/whisper')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Runs.omnilingual_asr_1b_levantine_custom_streaming_5minckpt.pipeline import (
    NOTEBOOK_PATH,
    SMOKE_NOTEBOOK_PATH,
    config_snapshot,
    ensure_run_layout,
    make_config,
    setup_logging,
)

RUN_SMOKE_TEST = True
EVAL_SAMPLE_CAP = 1
TRAIN_NUM_STEPS = 1
RUN_BASELINE_BEFORE_TRAIN = True
RUN_POST_TRAIN_EVAL = True

config = make_config(
    smoke_mode=RUN_SMOKE_TEST,
    eval_sample_cap=EVAL_SAMPLE_CAP,
    train_num_steps=TRAIN_NUM_STEPS,
    run_baseline_before_train=RUN_BASELINE_BEFORE_TRAIN,
    run_post_train_eval=RUN_POST_TRAIN_EVAL,
)

ensure_run_layout()
log_path = setup_logging()
print(json.dumps(config_snapshot(config), ensure_ascii=False, indent=2))
print('Notebook path:', SMOKE_NOTEBOOK_PATH if RUN_SMOKE_TEST else NOTEBOOK_PATH)
print('Log path:', log_path)


In [ ]:
from Runs.omnilingual_asr_1b_levantine_custom_streaming_5minckpt.pipeline import PREP_STATE_PATH, prepare_dataset

manifest_state = prepare_dataset(config)
selection_summary = manifest_state['selection_summary']
print(json.dumps(selection_summary, ensure_ascii=False, indent=2))
print('Prep cache:', PREP_STATE_PATH)
print('Loaded from cache:', selection_summary.get('loaded_from_cache', False))


In [ ]:
from Runs.omnilingual_asr_1b_levantine_custom_streaming_5minckpt.pipeline import run_predictions

baseline_test_metrics = None
if config.run_baseline_before_train:
    baseline_test_metrics = run_predictions(
        manifest_state['test_rows'],
        config,
        model_card=config.model_card,
        name='baseline_test_predictions',
    )
print(json.dumps(baseline_test_metrics, ensure_ascii=False, indent=2))


In [ ]:
from Runs.omnilingual_asr_1b_levantine_custom_streaming_5minckpt.pipeline import run_training

training_summary = run_training(config, manifest_state)
print(json.dumps(training_summary, ensure_ascii=False, indent=2))


In [ ]:
from Runs.omnilingual_asr_1b_levantine_custom_streaming_5minckpt.pipeline import model_asset_name, run_predictions

local_model_card = model_asset_name(config)
val_prediction_metrics = run_predictions(
    manifest_state['val_rows'],
    config,
    model_card=local_model_card,
    name='tuned_val_predictions',
) if config.run_post_train_eval else None

test_prediction_metrics = run_predictions(
    manifest_state['test_rows'],
    config,
    model_card=local_model_card,
    name='tuned_test_predictions',
) if config.run_post_train_eval else None

print('Validation metrics:')
print(json.dumps(val_prediction_metrics, ensure_ascii=False, indent=2))
print('Test metrics:')
print(json.dumps(test_prediction_metrics, ensure_ascii=False, indent=2))


In [ ]:
from Runs.omnilingual_asr_1b_levantine_custom_streaming_5minckpt.pipeline import write_integrity_report, write_summary_report

summary_report = write_summary_report(
    config,
    selection_summary,
    baseline_test_metrics,
    val_prediction_metrics,
    test_prediction_metrics,
    training_summary,
)
integrity_report = write_integrity_report(config, training_summary, summary_report)
print(json.dumps(summary_report, ensure_ascii=False, indent=2))
print(json.dumps(integrity_report, ensure_ascii=False, indent=2))
